# Did the boat strikes cut America's cocaine supply?

A walkthrough of the interrupted time series in this directory. The point of
the notebook is the middle section: the obvious analysis returns a confident
wrong answer, and a placebo test shows you why in about ten lines.

See [`README.md`](README.md) for the full write-up.

In [ ]:
# On Colab, fetch the project files first.
import os, sys, subprocess

if not os.path.exists("its_core.py"):
    subprocess.run(["pip", "install", "-q", "pandas", "numpy", "scipy", "matplotlib"], check=True)
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", "claude/time-series-drug-supply-boats-0ozk3v",
        "https://github.com/aflaxman/ai_assisted_research.git", "/content/repo",
    ], check=True)
    os.chdir("/content/repo/boat_strikes_drug_supply_its")
sys.path.insert(0, os.getcwd())
print(os.getcwd())

## 1. The intervention

69 numbered strikes destroyed 70 vessels between September 2025 and August
2026. Note where they happened — this is mostly not a Caribbean campaign.

In [ ]:
import pandas as pd

strikes = pd.read_csv("data/strikes.csv", parse_dates=["date"])
print(f"{strikes['n_strikes'].sum()} strikes on {len(strikes)} dates")
print(f"{strikes['vessels_struck'].sum()} vessels, {strikes['killed'].sum()}+ killed")
strikes.groupby("region")[["vessels_struck", "killed"]].sum()

## 2. The outcome, and its shape

CDC publishes only 12-month-ending rolling totals. That single fact drives
everything that follows.

In [ ]:
from run_analysis import load_outcomes, CAMPAIGN_START, fit_one
import its_core as ic

wide = load_outcomes("rolling12")
print(f"windows: {wide.index.min():%Y-%m} .. {wide.index.max():%Y-%m}")
wide.tail(8).round(0)

In [ ]:
# How much of each post-campaign window is actually post-campaign?
design = ic.build_design(pd.DatetimeIndex(wide.index), CAMPAIGN_START, effect="step")
lev = ic.window_leverage(design)
post = lev[lev["post_fraction"] > 0]
print(post.to_string(index=False))
print(f"\nmean leverage {post['post_fraction'].mean():.2f} "
      f"-> model error amplified ~{1 / post['post_fraction'].mean():.1f}x")

## 3. The obvious analysis, and the confident wrong answer

In [ ]:
fit = fit_one(wide["cocaine"], CAMPAIGN_START, "step")
eff = ic.relative_effect(fit)
print(f"cocaine step: {eff['pct_change']:+.1f}%  "
      f"(95% CI {eff['pct_lo']:+.1f} to {eff['pct_hi']:+.1f})")
print("Deaths UP after the strikes began. Is that real?")

## 4. The placebo test that settles it

Point the same estimator at a date when nothing happened. It should find
nothing.

In [ ]:
for year in (2022, 2023, 2024):
    placebo = pd.Timestamp(f"{year}-09-01")
    truncated = wide.loc[wide.index <= placebo + pd.DateOffset(months=6)]
    e = ic.relative_effect(fit_one(truncated["cocaine"], placebo, "step"))
    print(f"placebo {placebo:%b %Y}: {e['pct_change']:+6.1f}%  "
          f"({e['pct_lo']:+.1f} to {e['pct_hi']:+.1f})")

September 2023 "detects" a 17.7% drop. The estimator is reading curvature in
a decelerating trend, not interventions.

## 5. The fix: contrast against a drug the strikes cannot touch

Fentanyl and methamphetamine come overland from Mexico. Cocaine crosses open
water. Shared national shocks cancel in the difference.

In [ ]:
effects = {
    drug: ic.relative_effect(fit_one(wide[drug], CAMPAIGN_START, "step"))
    for drug in ("cocaine", "synthetic_opioids", "psychostimulants")
}
for drug, e in effects.items():
    print(f"{drug:20} {e['pct_change']:+6.1f}%  "
          f"({e['pct_lo']:+.1f} to {e['pct_hi']:+.1f})")

print()
for comp in ("synthetic_opioids", "psychostimulants"):
    c = ic.contrast_effects(effects["cocaine"], effects[comp])
    print(f"cocaine vs {comp:20} {c['pct_change']:+6.1f}%  "
          f"({c['pct_lo']:+.1f} to {c['pct_hi']:+.1f})")

## 6. What size of effect was even on the table?

A null is meaningless without knowing what you were looking for.

In [ ]:
import supply_arithmetic as sa

consistent = sa.grid()
consistent = consistent[consistent["consistent"]]["expected_death_change_pct"]
print(f"expected effect from tonnage: {consistent.min():.1f}% to "
      f"{consistent.max():.1f}% (median {consistent.median():.1f}%)")

curve = pd.read_csv("outputs/power_curve.csv")
print(curve.to_string(index=False))

The minimum detectable effect is about −23%; the median expected effect is
about −11%. The study was looking for something it could not have seen.

## Your turn

1. Change the comparator and see how far the contrast moves.
2. Add a lag: `fit_one(..., lag=4)`. Does the methamphetamine contrast survive?
3. Re-run once CDC publishes more windows and check whether the interval
   actually narrows — `power_analysis.py` predicts it barely will.